# 03 查看、选择与过滤

拿到一张表，日常动作无非四种：

1. **看**：有多少行、有没有缺失、分布长什么样；
2. **选列**：只留下我要的列；
3. **选行**：按条件筛出子集；
4. **改**：按条件批量改值。

本节把 `loc` / `iloc` / 布尔索引 / `query` 这套"取数语法"彻底理清，这是 pandas 使用频率最高的部分。

> 环境：`llm_sft`，从 `python/pandas` 目录启动。

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.unicode.east_asian_width", True)

DATA = Path("data")
if not DATA.exists():
    DATA = Path("python/pandas/data")

orders = pd.read_csv(DATA / "sales_orders.csv", parse_dates=["order_date"])
print("行数:", len(orders), "| 列:", orders.columns.tolist())
orders.head()

行数: 60 | 列: ['order_id', 'order_date', 'city', 'category', 'channel', 'unit_price', 'qty', 'amount', 'sales_rep']


,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
0,O001,2024-01-08,上海,食品,门店,95.9,4,383.6,王强
1,O002,2024-01-12,广州,家居,线上,453.4,4,1813.6,张伟
2,O003,2024-01-13,杭州,服饰,门店,120.7,3,362.1,王强
3,O004,2024-01-16,北京,家居,线上,758.4,4,3033.6,赵敏
4,O005,2024-01-17,广州,食品,门店,80.6,5,403.0,陈晨


## 1. 先体检：`head` / `info` / `describe` / `value_counts`

四个函数覆盖 80% 的"看看这数据长啥样"场景。

In [2]:
orders.head(3)          # 前 3 行

,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
0,O001,2024-01-08,上海,食品,门店,95.9,4,383.6,王强
1,O002,2024-01-12,广州,家居,线上,453.4,4,1813.6,张伟
2,O003,2024-01-13,杭州,服饰,门店,120.7,3,362.1,王强


In [3]:
orders.tail(3)          # 后 3 行

,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
57,O058,2024-06-16,杭州,服饰,线下,77.1,4,308.4,张伟
58,O059,2024-06-24,成都,数码,门店,1867.9,5,9339.5,张伟
59,O060,2024-06-25,成都,食品,线上,48.9,4,195.6,赵敏


In [4]:
orders.sample(3, random_state=0)   # 随机抽样：比 head 更能反映数据分布，random_state 保证可复现

,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
26,O027,2024-03-31,杭州,服饰,线下,272.9,4,1091.6,张伟
35,O036,2024-04-28,北京,服饰,线下,238.5,1,238.5,李娜
59,O060,2024-06-25,成都,食品,线上,48.9,4,195.6,赵敏


In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    60 non-null     str           
 1   order_date  60 non-null     datetime64[us]
 2   city        60 non-null     str           
 3   category    60 non-null     str           
 4   channel     60 non-null     str           
 5   unit_price  60 non-null     float64       
 6   qty         60 non-null     int64         
 7   amount      57 non-null     float64       
 8   sales_rep   58 non-null     str           
dtypes: datetime64[us](1), float64(2), int64(1), str(5)
memory usage: 6.0 KB


In [6]:
orders.describe()       # 数值列：count/mean/std/min/四分位/max

,order_date,unit_price,qty,amount
count,60,60.000000,60.000000,57.000000
mean,2024-04-04 13:36:00,370.350000,3.133333,1253.933333
min,2024-01-08 00:00:00,7.300000,1.000000,17.000000
25%,2024-03-01 12:00:00,77.175000,2.000000,154.400000
50%,2024-04-06 12:00:00,190.700000,3.000000,488.500000
75%,2024-05-17 00:00:00,537.125000,4.000000,1415.000000
max,2024-06-25 00:00:00,2221.600000,5.000000,9339.500000
std,NaN,447.509759,1.443474,2027.365526


In [7]:
orders.describe(include="all")   # 加上文本列：unique/top/freq

,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
count,60,60,60,60,60,60.000000,60.000000,57.000000,58
unique,60,NaN,6,4,3,NaN,NaN,NaN,5
top,O001,NaN,深圳,食品,线上,NaN,NaN,NaN,王强
freq,1,NaN,14,18,23,NaN,NaN,NaN,14
mean,NaN,2024-04-04 13:36:00,NaN,NaN,NaN,370.350000,3.133333,1253.933333,NaN
min,NaN,2024-01-08 00:00:00,NaN,NaN,NaN,7.300000,1.000000,17.000000,NaN
25%,NaN,2024-03-01 12:00:00,NaN,NaN,NaN,77.175000,2.000000,154.400000,NaN
50%,NaN,2024-04-06 12:00:00,NaN,NaN,NaN,190.700000,3.000000,488.500000,NaN
75%,NaN,2024-05-17 00:00:00,NaN,NaN,NaN,537.125000,4.000000,1415.000000,NaN
max,NaN,2024-06-25 00:00:00,NaN,NaN,NaN,2221.600000,5.000000,9339.500000,NaN


In [8]:
# 每一列有多少种取值、缺失多少、出现最多的值是什么
summary = pd.DataFrame({
    "dtype": orders.dtypes.astype(str),
    "缺失数": orders.isna().sum(),
    "缺失比例": (orders.isna().mean() * 100).round(1),
    "不同取值数": orders.nunique(),
})
summary

,dtype,缺失数,缺失比例,不同取值数
order_id,str,0,0.0,60
order_date,datetime64[us],0,0.0,50
city,str,0,0.0,6
category,str,0,0.0,4
channel,str,0,0.0,3
unit_price,float64,0,0.0,60
qty,int64,0,0.0,5
amount,float64,3,5.0,57
sales_rep,str,2,3.3,5


In [9]:
from IPython.display import display

display(orders["city"].value_counts().to_frame("订单数"))                    # 分类列的频次分布
display(orders["sales_rep"].value_counts(dropna=False).to_frame("订单数"))     # dropna=False 把缺失也统计进去

,订单数
city,
深圳,14
杭州,13
广州,12
北京,9
上海,7
成都,5


,订单数
sales_rep,
王强,14
张伟,13
赵敏,11
李娜,11
陈晨,9
NaN,2


## 2. 选列

| 写法 | 结果 | 说明 |
| --- | --- | --- |
| `df["col"]` | Series | 单列 |
| `df[["a", "b"]]` | DataFrame | 多列，**双层方括号** |
| `df.filter(like="amt")` | DataFrame | 按列名关键词/正则选 |
| `df.select_dtypes("number")` | DataFrame | 按 dtype 选 |

In [10]:
print(type(orders["amount"]).__name__)
print(type(orders[["city", "amount"]]).__name__)
orders[["order_id", "city", "amount"]].head(3)

Series
DataFrame


,order_id,city,amount
0,O001,上海,383.6
1,O002,广州,1813.6
2,O003,杭州,362.1


In [11]:
display(orders.filter(like="price").head(3))       # 列名里带 price 的列
display(orders.select_dtypes("number").head(3))    # 只要数值列

,unit_price
0,95.9
1,453.4
2,120.7


,unit_price,qty,amount
0,95.9,4,383.6
1,453.4,4,1813.6
2,120.7,3,362.1


## 3. 选行：`[]` / `.loc` / `.iloc`

三种方式，区别只在"按什么选、含不含结尾"：

| 写法 | 依据 | 结尾 | 典型用途 |
| --- | --- | --- | --- |
| `df.loc[标签]` | 行标签（index 的值） | **包含** | 按 ID、按日期区间取数 |
| `df.iloc[位置]` | 行号（0 开始） | **不包含** | 取前 N 行、按位置切 |
| `df[布尔条件]` | 条件 | — | 筛选（最常用） |

记住一句话：**要可预测就用 `loc` / `iloc`，不要用裸 `df[...]` 取行。**

In [12]:
# 位置切片：第一行到第三行，不含结尾
print("iloc[0:3] 的 order_id:", orders.iloc[0:3]["order_id"].tolist())

# 标签切片：index 还是默认的 0..59，所以 loc[0:3] 会包含 3
print("loc[0:3]  的 order_id:", orders.loc[0:3]["order_id"].tolist())

# 单行：loc 返回 Series（行），iloc 同理
orders.loc[1]

iloc[0:3] 的 order_id: ['O001', 'O002', 'O003']
loc[0:3]  的 order_id: ['O001', 'O002', 'O003', 'O004']


order_id                     O002
order_date    2024-01-12 00:00:00
city                         广州
category                     家居
channel                      线上
unit_price                  453.4
qty                             4
amount                     1813.6
sales_rep                    张伟
Name: 1, dtype: object

In [13]:
# 换成业务主键当索引之后，loc 才是真正好用的
by_id = orders.set_index("order_id").sort_index()
print("索引:", by_id.index[:3].tolist())
print(by_id.loc["O005", ["city", "amount"]])

索引: ['O001', 'O002', 'O003']
city       广州
amount    403.0
Name: O005, dtype: object


## 4. 布尔筛选

核心只有一句：**把条件算成一个 True/False 的 Series，再用它筛行。**

- 条件之间用 `&`（且）/ `|`（或）/ `~`（非），**每个条件必须加括号**；
- 不能用 Python 的 `and` / `or`。

In [14]:
mask = orders["amount"] > 1000
print("满足条件的行数:", mask.sum())
orders[mask].head(3)

满足条件的行数: 17


,order_id,order_date,city,category,channel,unit_price,qty,amount,sales_rep
1,O002,2024-01-12,广州,家居,线上,453.4,4,1813.6,张伟
3,O004,2024-01-16,北京,家居,线上,758.4,4,3033.6,赵敏
5,O006,2024-01-17,杭州,家居,线上,632.3,4,2529.2,张伟


In [15]:
# 多条件组合
big_orders = orders[(orders["amount"] > 800) & (orders["city"].isin(["北京", "上海"]))]
print("大额订单里的京沪订单:", len(big_orders))
big_orders[["order_id", "city", "amount"]].head()

大额订单里的京沪订单: 5


,order_id,city,amount
3,O004,北京,3033.6
11,O012,上海,7109.0
15,O016,北京,996.5
46,O047,北京,4368.5
54,O055,上海,910.5


In [16]:
# 用 and 会直接报错：DataFrame 的真值不明确
try:
    orders[(orders["amount"] > 800) and (orders["city"] == "北京")]
except ValueError as e:
    print("用 and 报错:", str(e)[:80])
    print("→ 多条件请用 & | ~，并且每个条件加括号")

用 and 报错: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any
→ 多条件请用 & | ~，并且每个条件加括号


In [17]:
# 常用条件函数：isin / between / str.contains / isna
print("数码或家居 :", orders["category"].isin(["数码", "家居"]).sum())
print("金额 500-1500:", orders["amount"].between(500, 1500).sum())
print("城市含'州'  :", orders["city"].str.contains("州", na=False).sum())
print("销售代表缺失:", orders["sales_rep"].isna().sum())

数码或家居 : 26
金额 500-1500: 14
城市含'州'  : 25
销售代表缺失: 2


### `query`：把条件写成字符串

条件很长时，`query` 更接近人话。外部变量用 `@变量名` 引用；**中文列名要用反引号**包起来。

In [18]:
min_amount = 800
orders.query("amount > @min_amount and city in ['北京', '上海']")[["order_id", "city", "amount"]].head()

,order_id,city,amount
3,O004,北京,3033.6
11,O012,上海,7109.0
15,O016,北京,996.5
46,O047,北京,4368.5
54,O055,上海,910.5


In [19]:
# 中文列名 + query
cn = pd.DataFrame({"城市": ["北京", "上海"], "金额": [1200, 300]})
cn.query("`金额` > 500")

,城市,金额
0,北京,1200


## 5. 按条件改值

三个层次：

- `df.loc[条件, 列] = 值`：最常用，精确改；
- `mask(条件, 值)`：条件为真处替换，返回新表；
- `where(条件, 值)`：条件为真处保留，否则替换。

In [20]:
d = orders.copy()
d.loc[d["amount"] > 3000, "channel"] = "大额直签"    # 改这一列
d.loc[d["amount"] > 3000, ["channel", "sales_rep"]] = ["大额直签", "张伟"]  # 改多列
d[d["amount"] > 3000][["order_id", "amount", "channel", "sales_rep"]].head()

,order_id,amount,channel,sales_rep
3,O004,3033.6,大额直签,张伟
11,O012,7109.0,大额直签,张伟
16,O017,8886.4,大额直签,张伟
40,O041,5068.0,大额直签,张伟
46,O047,4368.5,大额直签,张伟


In [21]:
amt = orders["amount"]
print("mask：>2000 的替换成 2000")
print(amt.mask(amt > 2000, 2000).head(6).tolist())

print("where：<=2000 的保留，其余变 NaN")
print(amt.where(amt <= 2000).head(6).tolist())

mask：>2000 的替换成 2000
[383.6, 1813.6, 362.1, 2000.0, 403.0, 2000.0]
where：<=2000 的保留，其余变 NaN
[383.6, 1813.6, 362.1, nan, 403.0, nan]


## 6. 单值访问：`at` / `iat`

只要一个格子时，`at`（标签）/ `iat`（位置）比 `loc` / `iloc` 更快，也更能表达意图。

In [22]:
print(orders.at[0, "amount"])
print(orders.iat[0, 5])
orders.loc[0, ["order_id", "city", "amount"]]   # 相比 loc 取一行的 Series，这里更清楚

383.6
95.9


order_id     O001
city         上海
amount      383.6
Name: 0, dtype: object

## 7. 两层索引（MultiIndex）怎么取

`set_index` 传多个列名就会得到两层索引，取数时把标签写成元组。
第 07 节还会用到它。

In [23]:
mi = orders.set_index(["city", "channel"]).sort_index()
print("索引层数:", mi.index.nlevels, "| 名字:", mi.index.names)
print()
print(mi.loc[("北京", "线上"), ["order_id", "amount"]].head())
print()
print("用 xs 取某一层：", mi.xs("上海", level="city").shape)

索引层数: 2 | 名字: ['city', 'channel']

             order_id  amount
city channel                 
北京 线上        O004  3033.6
     线上        O016   996.5
     线上        O047  4368.5
     线上        O051    26.6

用 xs 取某一层： (7, 7)


## 8. 小练习

从 `orders` 里筛出满足下面所有条件的订单，并按金额从高到低列出前 5 条：

1. 下单日期在 2024-03-01 之后；
2. 金额大于 500；
3. 品类是"数码"或"家居"；
4. 只保留 `order_id, order_date, city, category, amount` 五列；
5. 额外加一列 `tier`：金额 ≥ 2000 为 "A"，否则 "B"。

In [24]:
result = (
    orders[
        (orders["order_date"] > "2024-03-01")
        & (orders["amount"] > 500)
        & (orders["category"].isin(["数码", "家居"]))
    ][["order_id", "order_date", "city", "category", "amount"]]
    .sort_values("amount", ascending=False)
    .head(5)
    .assign(tier=lambda x: np.where(x["amount"] >= 2000, "A", "B"))
)
result

,order_id,order_date,city,category,amount,tier
58,O059,2024-06-24,成都,数码,9339.5,A
16,O017,2024-03-06,杭州,数码,8886.4,A
40,O041,2024-05-09,深圳,数码,5068.0,A
46,O047,2024-05-20,北京,数码,4368.5,A
29,O030,2024-04-05,广州,家居,2735.5,A


In [25]:
# 等价写法：用 query，条件长的时候更易读
result2 = orders.query(
    "order_date > '2024-03-01' and amount > 500 and category in ['数码', '家居']"
).nlargest(5, "amount")[["order_id", "order_date", "city", "category", "amount"]]
result2

,order_id,order_date,city,category,amount
58,O059,2024-06-24,成都,数码,9339.5
16,O017,2024-03-06,杭州,数码,8886.4
40,O041,2024-05-09,深圳,数码,5068.0
46,O047,2024-05-20,北京,数码,4368.5
29,O030,2024-04-05,广州,家居,2735.5


## 9. 小结与常见坑

**规则**

1. 选列：单列 `df["c"]`，多列 `df[["a","b"]]`。
2. 选行：`loc` 按标签（含结尾），`iloc` 按位置（不含结尾），筛选用布尔 Series。
3. 多条件：`&` / `|` / `~`，**每个条件加括号**，不要用 `and` / `or`。
4. 改值：`df.loc[条件, 列] = 值`；不想改原表先 `.copy()`。
5. `query` 里外部变量用 `@x`，中文列名用反引号。

**常见坑**

- `df[0:3]` 这种裸切片只对整数索引"看起来像按位置"，换索引后就变了 → 统一用 `iloc` / `loc`。
- `NaN == NaN` 是 False，判断缺失只能用 `isna()` / `notna()`。
- 链式赋值 `df[cond]["col"] = v` 在 pandas 3.0 下不生效（Copy-on-Write），会给出 `ChainedAssignmentError` 警告。
- `str.contains` 默认会把缺失当缺失，返回 NaN；筛选时记得 `na=False`。
- `between` 默认两端都包含（`inclusive="both"`）。